# Weather Agent Debugging Report

This notebook documents the debugging process for **Assignment 1: Weather Agent Debugging**.

The goal of the assignment is to repair a broken LangGraph weather agent so that it can:

- detect the user's location using IP-based geolocation
- fetch current weather data using latitude and longitude
- generate a readable weather report
- handle invalid API responses properly
- pass multiple testing scenarios

Rather than only showing the final working code, this notebook explains the debugging reasoning behind each issue, the cause of the failure, the implemented fix, and how the behavior was validated.

## 1. Debugging Approach

The debugging process followed a systematic workflow:

1. Run the program before making assumptions.
2. Read the exact error message.
3. Identify where the failure happened.
4. Separate environment errors from application logic errors.
5. Fix one issue at a time.
6. Rerun the program or tests after each fix.
7. Add tests for both working cases and failure cases.

This approach is important because debugging is not just about changing code until it works. Each fix should be connected to a clear cause.

## 2. Project Architecture

The weather agent is built using LangGraph. The workflow is modeled as a graph where each node performs one responsibility and passes updated state to the next node.

The expected flow is:

```text
START
  ↓
fetch_location_data
  ↓
fetch_weather_data
  ↓
generate_weather_info
  ↓
END
```

The shared state contains:

| State Field | Purpose |
|---|---|
| `name` | User's name for personalized output |
| `location_data` | Location information from the IP geolocation API |
| `weather_data` | Weather information from the weather API |
| `weather_info` | Final formatted weather report |

The important concept is that each node depends on state values produced by earlier nodes. Therefore, the order of graph execution matters.

In [6]:
from graph import weather_agent

print("Weather agent imported successfully.")
print("Compiled graph object:", type(weather_agent))

c:\Users\LENOVO\miniconda3\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


Weather agent imported successfully.
Compiled graph object: <class 'langgraph.graph.state.CompiledStateGraph'>


## 3. Bug: Missing Dependency Installation

### Error observed

```text
ModuleNotFoundError: No module named 'langgraph'
```

### Reasoning

The error occurred before the program reached the weather logic. Python failed while importing `StateGraph` from LangGraph.

This means the issue was not yet with the graph structure or API calls. The immediate problem was the Python environment.

### Fix

The virtual environment was activated and dependencies were installed using:

```bash
pip install -r requirements.txt
```

### Why this fix is correct

The project depends on external packages such as LangGraph, Pydantic, and Requests. These packages must be installed in the active environment before the program can run.

In [8]:
%pip install -r weather_agent/requirements.txt

  Using cached langchain-0.3.26-py3-none-any.whl.metadata (7.8 kB)
  Using cached langchain_core-0.3.75-py3-none-any.whl.metadata (5.7 kB)
  Using cached langgraph-0.6.4-py3-none-any.whl.metadata (6.8 kB)
  Using cached pydantic-2.11.5-py3-none-any.whl.metadata (67 kB)
  Using cached langchain_text_splitters-0.3.11-py3-none-any.whl.metadata (1.8 kB)
  Using cached sqlalchemy-2.0.49-cp313-cp313-win_amd64.whl.metadata (9.8 kB)
  Using cached pydantic_core-2.33.2-cp313-cp313-win_amd64.whl.metadata (6.9 kB)
  Using cached langgraph_checkpoint-2.1.2-py3-none-any.whl.metadata (4.2 kB)
  Using cached langgraph_prebuilt-0.6.5-py3-none-any.whl.metadata (4.5 kB)
  Using cached langgraph_sdk-0.2.15-py3-none-any.whl.metadata (1.6 kB)
  Using cached greenlet-3.5.0-cp313-cp313-win_amd64.whl.metadata (3.8 kB)
Using cached langchain-0.3.26-py3-none-any.whl (1.0 MB)
Using cached langchain_core-0.3.75-py3-none-any.whl (443 kB)
Using cached pydantic-2.11.5-py3-none-any.whl (444 kB)
Using cached langgraph

In [9]:
%pip show langgraph

Name: langgraph
Version: 0.6.4
Summary: Building stateful, multi-actor applications with LLMs
Home-page: 
Author: 
Author-email: 
License-Expression: MIT
Location: c:\Users\LENOVO\miniconda3\Lib\site-packages
Requires: langchain-core, langgraph-checkpoint, langgraph-prebuilt, langgraph-sdk, pydantic, xxhash
Required-by: 
Note: you may need to restart the kernel to use updated packages.


## 4. Bug: Invalid Pydantic Configuration Field

### Error observed

```text
PydanticUserError: A non-annotated attribute was detected: TEMP_MIN = 'Needs Debugging'
```

### Reasoning

The `Config` class inherits from Pydantic's settings model. Pydantic requires configuration fields to have proper type annotations.

The field below was an invalid placeholder:

```python
TEMP_MIN = "Needs Debugging"
```

It was also not a meaningful configuration value for the weather agent.

### Fix

The invalid placeholder was removed. The code instead uses properly typed temperature thresholds:

```python
TEMP_COLD: float = 10.0
TEMP_COOL: float = 18.0
TEMP_COMFORTABLE: float = 26.0
TEMP_WARM: float = 32.0
```

### Why this fix is correct

Temperature classification should be based on numerical thresholds, not a debugging placeholder string.

## 5. Bug: Incorrect Python Entry Point

### Problem observed

Running:

```bash
python main.py
```

initially produced no output.

### Reasoning

When a Python file runs without output and without an error, one possible cause is that the expected function was defined but never called.

In this case, the `main()` function was not being executed because the script entry point was incorrect.

### Fix

The entry point was corrected to:

```python
if __name__ == "__main__":
    main()
```

### Why this fix is correct

This condition ensures that `main()` runs when the file is executed directly using `python main.py`.

## 6. Bug: Incorrect LangGraph Flow

### Problem observed

The graph attempted to generate weather information before weather data had been fetched.

### Reasoning

The weather report requires two pieces of information:

1. location data
2. weather data

However, the graph skipped the `fetch_weather_data` node and connected location fetching directly to report generation.

This caused the workflow to fail because `generate_weather_info` depended on missing weather data.

### Fix

The graph edges were updated to follow the correct dependency order:

```python
builder.add_edge(START, "fetch_location_data")
builder.add_edge("fetch_location_data", "fetch_weather_data")
builder.add_edge("fetch_weather_data", "generate_weather_info")
builder.add_edge("generate_weather_info", END)
```

### Why this fix is correct

The weather API needs latitude and longitude from the location node. The final report then needs both location and weather data. The corrected graph reflects these dependencies.

## 7. Bug: Missing State Update

### Error observed

```text
Location data not available for weather fetch
```

### Reasoning

The location API response was fetched and validated, but the result was not stored in the shared LangGraph state.

Because LangGraph nodes communicate through state, the next node could not access the location data.

### Fix

The location data was saved into state before returning from the node:

```python
state["location_data"] = location_data
```

### Why this fix is correct

`fetch_weather_data` depends on `state["location_data"]` to get latitude and longitude. Without explicitly saving this value, the graph cannot pass data between nodes.

## 8. Bug: Inconsistent API Field Names

### Error observed

```text
Missing data field for weather info generation: 'country_name'
```

### Reasoning

Some parts of the code used `country`, while other parts expected `country_name`.

This mismatch caused the final formatter to fail even though location data existed.

### Fix

The location field was standardized to use:

```python
country_name
```

This field name was used consistently in:

- location validation
- mocked test data
- weather report generation

### Why this fix is correct

A pipeline should use consistent field names from input validation to output formatting. Otherwise, data can exist in state but still be inaccessible to later code.

## 9. Bug: Broken Temperature Classification Logic

### Broken code

```python
if temp_celsius:
    return config.TEMP_MIN
```

### Reasoning

The condition `if temp_celsius:` only checks whether the value is truthy. Most nonzero temperatures such as `7.0`, `24.0`, and `29.0` evaluate to `True`.

This caused the function to return immediately before reaching the actual threshold comparisons. It also referenced `TEMP_MIN`, which had already been removed.

### Fix

The function was rewritten to compare the temperature against numeric thresholds:

```python
def classify_temperature(temp_celsius: float) -> str:
    if temp_celsius < config.TEMP_COLD:
        return "cold"
    elif temp_celsius < config.TEMP_COOL:
        return "cool"
    elif temp_celsius < config.TEMP_COMFORTABLE:
        return "comfortable"
    elif temp_celsius < config.TEMP_WARM:
        return "warm"
    else:
        return "hot"
```

### Why this fix is correct

Temperature classification is a range-based decision. It should compare the temperature against thresholds rather than checking if the value is truthy.

In [10]:
from components.helper_functions import classify_temperature

sample_temperatures = [7.0, 15.0, 24.0, 29.0, 35.0]

for temp in sample_temperatures:
    print(f"{temp}C -> {classify_temperature(temp)}")

7.0C -> cold
15.0C -> cool
24.0C -> comfortable
29.0C -> warm
35.0C -> hot


## 10. Bug: Incorrect Output Formatting and Variable Interpolation

### Problem observed

The report displayed:

```text
Wind: 14.0 wind_unit
```

instead of showing the actual unit:

```text
Wind: 14.0 km/h
```

### Reasoning

The program printed the literal string `wind_unit` instead of inserting the value of the `wind_unit` variable.

### Fix

The formatted string was corrected to:

```python
f"- Wind: {windspeed} {wind_unit}"
```

### Why this fix is correct

Formatted strings require variables to be wrapped in curly braces. Without curly braces, Python treats the text as a literal string.

## 11. Live Testing Limitation: API Rate Limiting

### Error observed

```text
429 Client Error: Too Many Requests
```

### Reasoning

This error occurred when calling the IP geolocation API. The application reached the external service, but the service temporarily rejected the request due to rate limiting.

This is not the same as a graph logic error. It is an external API reliability issue.

### Response

To avoid depending entirely on external API availability, the agent was tested using mocked API responses.

### Why this matters

External services can fail due to rate limits, downtime, network issues, or quota restrictions. Reliable tests should validate application logic without depending on live API behavior every time.

## 12. Testing Strategy

The testing strategy uses mocked API responses instead of live API calls.

This allows the tests to verify the LangGraph workflow deterministically.

The test suite covers:

| Test Case | Purpose |
|---|---|
| Daytime clear weather | Verifies normal successful output during daytime |
| Nighttime rainy weather | Verifies weather code mapping, greeting, cold classification, and wind formatting |
| Missing location field | Verifies error handling for invalid location API responses |

This gives coverage for both working cases and an error-handling case.

In [14]:
!cd weather_agent && python -m unittest test_weather_agent.py

...
----------------------------------------------------------------------
Ran 3 tests in 0.031s

OK


## 13. Working Case: Daytime Clear Weather

This test uses mocked API responses to simulate a successful daytime weather report.

Expected behavior:

- the graph fetches mocked location data
- the graph fetches mocked weather data
- the final report greets the user with a daytime greeting
- the weather code is converted into a readable description
- the temperature is classified correctly
- wind speed and unit are displayed correctly

In [15]:
from unittest.mock import patch
from graph import weather_agent

class MockResponse:
    def __init__(self, payload):
        self.payload = payload

    def raise_for_status(self):
        return None

    def json(self):
        return self.payload

location_response = {
    "city": "Baguio City",
    "region": "Cordillera Administrative Region",
    "country_name": "Philippines",
    "latitude": 16.4023,
    "longitude": 120.5960,
    "utc_offset": "+08:00",
    "timezone": "Asia/Manila",
}

weather_response = {
    "current_weather_units": {
        "temperature": "C",
        "windspeed": "km/h",
    },
    "current_weather": {
        "time": "2026-05-06T10:30",
        "temperature": 24.0,
        "windspeed": 8.5,
        "winddirection": 120,
        "is_day": 1,
        "weathercode": 0,
    },
}

with patch("components.nodes.requests.get", side_effect=[MockResponse(location_response), MockResponse(weather_response)]):
    final_state = weather_agent.invoke({
        "name": "Aivann",
        "location_data": None,
        "weather_data": None,
        "weather_info": None,
    })

print(final_state["weather_info"])

Time: 10:30 UTC | 18:30 (UTC+08:00)

Good morning, Aivann!

Your current location: Baguio City, Cordillera Administrative Region, Philippines

Current weather conditions:
• Clear sky
• Temperature: 24.0C (comfortable)
• Wind: 8.5 km/h


## 14. Working Case: Nighttime Rainy Weather

This test verifies that the agent can handle a different location, a nighttime greeting, rainy weather, and cold temperature classification.

Expected behavior:

- the greeting should be `Good evening`
- weather code `63` should map to `Moderate rain`
- temperature `7.0C` should be classified as `cold`
- wind speed should include the correct unit

In [16]:
location_response = {
    "city": "London",
    "region": "England",
    "country_name": "United Kingdom",
    "latitude": 51.5072,
    "longitude": -0.1276,
    "utc_offset": "+00:00",
    "timezone": "Europe/London",
}

weather_response = {
    "current_weather_units": {
        "temperature": "C",
        "windspeed": "km/h",
    },
    "current_weather": {
        "time": "2026-05-06T10:30",
        "temperature": 7.0,
        "windspeed": 14.0,
        "winddirection": 120,
        "is_day": 0,
        "weathercode": 63,
    },
}

with patch("components.nodes.requests.get", side_effect=[MockResponse(location_response), MockResponse(weather_response)]):
    final_state = weather_agent.invoke({
        "name": "Aivann",
        "location_data": None,
        "weather_data": None,
        "weather_info": None,
    })

print(final_state["weather_info"])

Time: 10:30 UTC | 10:30 (UTC+00:00)

Good evening, Aivann!

Your current location: London, England, United Kingdom

Current weather conditions:
• Moderate rain
• Temperature: 7.0C (cold)
• Wind: 14.0 km/h


## 15. Error-Handling Case: Missing Location Field

This test verifies that the agent detects invalid location API responses.

The location response intentionally omits `country_name`. Since this field is required for the final weather report, the agent should raise a clear error instead of silently continuing with incomplete data.

In [17]:
bad_location_response = {
    "city": "Baguio City",
    "region": "Cordillera Administrative Region",
    # country_name is intentionally missing
    "latitude": 16.4023,
    "longitude": 120.5960,
    "utc_offset": "+08:00",
    "timezone": "Asia/Manila",
}

weather_response = {
    "current_weather_units": {
        "temperature": "C",
        "windspeed": "km/h",
    },
    "current_weather": {
        "time": "2026-05-06T10:30",
        "temperature": 24.0,
        "windspeed": 8.5,
        "winddirection": 120,
        "is_day": 1,
        "weathercode": 0,
    },
}

try:
    with patch("components.nodes.requests.get", side_effect=[MockResponse(bad_location_response), MockResponse(weather_response)]):
        weather_agent.invoke({
            "name": "Aivann",
            "location_data": None,
            "weather_data": None,
            "weather_info": None,
        })
except Exception as error:
    print("Handled error:")
    print(error)

Handled error:
Invalid location data received: Missing required field: country_name


## 16. Final Test Result

The final unit test run produced:

```text
Ran 3 tests

OK
```

This confirms that:

- the LangGraph workflow executes in the correct order
- state propagation works correctly
- weather data is fetched only after location data is available
- weather reports are generated correctly
- invalid location responses are handled properly
- tests do not depend on live API availability

## 17. Summary of Fixes

| Issue | Root Cause | Fix |
|---|---|---|
| Missing dependency | Required packages were not installed | Installed dependencies from `requirements.txt` |
| Invalid Pydantic field | `TEMP_MIN` placeholder had no type annotation | Removed placeholder and used typed threshold fields |
| Script produced no output | `main()` was not executed | Fixed `if __name__ == "__main__"` |
| Incorrect graph flow | `fetch_weather_data` was skipped | Added the missing graph edge sequence |
| Missing location data | Location response was not saved to state | Added `state["location_data"] = location_data` |
| Field name mismatch | Mixed use of `country` and `country_name` | Standardized to `country_name` |
| Broken temperature logic | Truthy check returned too early | Replaced with threshold comparisons |
| Incorrect wind output | Literal text `wind_unit` was printed | Used `{wind_unit}` interpolation |
| API rate limiting | External service rejected requests | Used mocked tests for reliable validation |

## 18. Conclusion

The weather agent was successfully debugged and tested.

The final implementation demonstrates:

- correct LangGraph workflow construction
- proper state propagation between nodes
- consistent API response validation
- reliable temperature classification
- clean weather report formatting
- error handling for invalid API responses
- unit testing with mocked external services

The debugging process showed that the main issues were not isolated to one file. They involved configuration, graph structure, state propagation, helper logic, output formatting, and test reliability.

By fixing these issues systematically, the agent became functional, testable, and easier to maintain.